# DATA MATCH — воспроизведение оценок полевых игроков с нуля

Ноутбук запускает весь конвейер методики (см. `docs/METHODOLOGY.md`) и выдаёт `outputs/test_ratings.csv`:
одна строка на полевого игрока тестовых матчей «Авангарда» с оценкой и границами 95%-интервала.

**Шаги:** L0 подготовка событий и смен → L1 xG → L2 ценность ситуации V → L3 вклад действий δ →
L5 стягивание по амплуа (итог O1, страховочный O0) → L6 интервалы (бутстрэп моделей ценности, B = 200).

**Требования:** Python 3.12, зависимости из `requirements.lock.txt` в `.venv`, данные в
`../DataMatch_Аналитический_трек/data`. CPU, около 15 ГБ ОЗУ. Ориентировочное время с нуля:
L0 ≈ 5 мин, L1 ≈ 1–1,5 ч, L2 ≈ 1–2 ч, L3/L5 — минуты, L6 (B = 200, 2 процесса) — много часов.
Повторный запуск берёт готовые этапы из кэша по ключу (данные, конфигурация, код, входы).

Данные хакатона не публикуются и не передаются внешним сервисам.

## Быстрая проверка готового локального расчёта

Следующие ячейки читают `outputs/test_ratings.csv`, `outputs/card/club_ratings.csv` и
`outputs/validation_report.json`. Эти файлы исключены из Git и появятся после локального расчёта
конвейера и отдельного запуска `scripts/validation_report.py`.
Если результатов ещё нет, начните с раздела «Подготовка к полному воспроизведению» ниже;
после расчёта вернитесь сюда. Для этих ячеек нужны только Python и pandas.
Сверка SHA-256 с эталоном носит диагностический характер: на другой машине возможен иной выбор модели.


In [ ]:
import hashlib
import json
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent                      # корень проекта
OUT = ROOT / "outputs"
REFERENCE_SHA = "eec6df1d194787ac3ac18dc6022f696a72e2a5e5ca98fa3884a0287157a8efaa"  # сдаваемая версия A

required = [OUT / "test_ratings.csv", OUT / "card" / "club_ratings.csv", OUT / "validation_report.json"]
missing = [str(p.relative_to(ROOT)) for p in required if not p.exists()]
assert not missing, (
    "Нет локальных результатов: " + ", ".join(missing)
    + ". Начните с раздела подготовки ниже, затем запустите scripts/validation_report.py."
)

sha = hashlib.sha256((OUT / "test_ratings.csv").read_bytes()).hexdigest()
print("sha256 test_ratings.csv:", sha)
print("совпадает с эталоном сдаваемой версии A:", sha == REFERENCE_SHA)

ratings = pd.read_csv(OUT / "test_ratings.csv")
print(len(ratings), "полевых игроков | амплуа:", ratings.role.value_counts().to_dict(),
      "| надёжность:", ratings.reliability.value_counts().to_dict())
club = pd.read_csv(OUT / "card" / "club_ratings.csv")[["player_id", "is_avangard"]]
avg = ratings.merge(club, on="player_id").query("is_avangard")
print(f"«Авангард»: {len(avg)} полевых | диапазон целиком выше среднего амплуа: {(avg.lo95 > 0).sum()}",
      f"| ниже: {(avg.hi95 < 0).sum()} | отличие не подтверждено: {((avg.lo95 <= 0) & (avg.hi95 >= 0)).sum()}")
ratings.sort_values("rating", ascending=False).head(10)

In [ ]:
report = json.loads((OUT / "validation_report.json").read_text())
rel = report["reliability_train"]["ratings"]
rows = [("Наша оценка (O1)", "O1"), ("Броски / 60", "ishots_60_raw"), ("Личный xG / 60", "ixg_60_raw"),
        ("Очки / 60", "points_60_raw"), ("Разница xG на льду, 5 на 5", "xgd_5v5_60_raw"),
        ("Разница голов на льду, 5 на 5", "gd_5v5_60_raw")]
table = pd.DataFrame([{"показатель": name, "повторяемость": round(rel[k]["pooled"]["spearman_sb"], 2),
                       "95%-интервал": [round(x, 2) for x in rel[k]["pooled"]["spearman_sb_ci"]]} for name, k in rows])
cov = report["calibration"]["predictive_coverage_train"]["pooled"]
print("Повторяемость: половина матчей против другой (Спирмен–Браун, игроки train)")
print(f"Попадание результата другой половины в 95%-диапазон: {cov['coverage']:.1%} "
      f"[{cov['coverage_ci'][0]:.1%}; {cov['coverage_ci'][1]:.1%}]")
table

## Подготовка к полному воспроизведению

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent                      # корень проекта
PY = ROOT / ".venv" / "bin" / "python"
# Пути данных читаются из той же конфигурации, что использует CLI.
sys.path.insert(0, str(ROOT / "src"))
from datamatch.config import load_config

cfg = load_config()
DATA = cfg.paths.data_dir
TEST_DATA = cfg.paths.test_dir
OUT = ROOT / "outputs"

# FROM_SCRATCH = True пересчитывает все этапы, игнорируя кэш (--force-all на первом шаге и --force далее).
FROM_SCRATCH = False
assert PY.exists(), f"нет окружения {PY}: python3 -m venv .venv && .venv/bin/pip install -r requirements.lock.txt"
assert DATA.exists(), f"нет данных хакатона в {DATA}: настройте configs/default.yaml"
assert TEST_DATA.exists(), f"нет тестовых данных в {TEST_DATA}: настройте configs/default.yaml"
print("корень:", ROOT)

In [ ]:
def run(*args: str) -> None:
    """Запускает команду конвейера; при провале гейта (код 2) или ошибке останавливает ноутбук."""
    env = dict(os.environ, PYTHONPATH=str(ROOT / "src"))
    cmd = [str(PY), "-m", "datamatch.cli", *args]
    print("$", " ".join(cmd[1:]), flush=True)
    proc = subprocess.run(cmd, cwd=ROOT, env=env, capture_output=True, text=True)
    tail = "\n".join((proc.stdout + proc.stderr).strip().splitlines()[-8:])
    print(tail)
    if proc.returncode != 0:
        raise RuntimeError(f"{args[0]}: код {proc.returncode} (2 — провален гейт качества, см. artifacts/*.failed.json)")

force = ["--force"] if FROM_SCRATCH else []

## L0 — события, смены, время на льду, разбиения

In [ ]:
run("prepare", *(["--force-all"] if FROM_SCRATCH else []))
quality = json.loads((OUT / "data_quality.json").read_text())
print("статус L0:", quality["status"])

## L1 — ожидаемые голы (турнир моделей на внутренней валидации)

In [ ]:
run("xg", *force)
xg = json.loads((ROOT / "artifacts" / "xg_selected.json").read_text())
pd.DataFrame({k: {m: v[m] for m in ("logloss", "auc_all", "ece")} for k, v in xg["all_candidates"].items()}).T.round(4)

## L2 — ценность ситуации V(s): турнир, гейт калибровки, итоговая подгонка на всём train

In [ ]:
run("value", *force)
v = json.loads((ROOT / "artifacts" / "value_selected.json").read_text())
print("выбрано:", v["selected"], "| гейт:", v["gate_l2"], "| итоговая подгонка:", v["final_fit"]["matches"], "матчей")

### Проверка L2 и замороженный внешний аудит

In [ ]:
subprocess.run([str(PY), "scripts/verify_l2_artifacts.py"], cwd=ROOT, env=dict(os.environ, PYTHONPATH=str(ROOT / "src")), check=True)
subprocess.run([str(PY), "scripts/audit_frozen_layers.py"], cwd=ROOT, env=dict(os.environ, PYTHONPATH=str(ROOT / "src")), check=True)

## L3 — вклад действий δ и таблица «игрок × матч»

In [ ]:
run("contrib", *force)

## L5 — стягивание по амплуа, итог O1 (страховочный O0)

In [ ]:
run("shrink", *force)

### Проверка: наша оценка против базовых показателей (устойчивость, прогноз будущей игры)

In [ ]:
subprocess.run([str(PY), "scripts/validation_report.py"], cwd=ROOT, env=dict(os.environ, PYTHONPATH=str(ROOT / "src")), check=True)

## L6 — интервалы и надёжность, сдаваемый файл (B = 200)

In [ ]:
run("ratings", "--mode", "full", *force)
# правило отображения (METHODOLOGY 7.7): TOI < 60 мин → LOW «мало минут»; рейтинг и интервал те же
env = dict(os.environ, PYTHONPATH=str(ROOT / "src"))
subprocess.run([str(PY), "scripts/apply_display_rules.py", "--mode", "full"], cwd=ROOT, env=env, check=True)
subprocess.run([str(PY), "scripts/apply_display_rules.py", "--mode", "full", "--check"], cwd=ROOT, env=env, check=True)

## Результат: `outputs/test_ratings.csv`

In [ ]:
ratings = pd.read_csv(OUT / "test_ratings.csv")
assert ratings.player_id.is_unique, "одна строка на игрока"
assert (ratings.lo95 < ratings.hi95).all() and ratings.rating.between(ratings.lo95, ratings.hi95).all()
print(len(ratings), "игроков;", ratings.reliability.value_counts().to_dict())
ratings.sort_values("rating", ascending=False).head(15)

In [ ]:
meta = json.loads((OUT / "test_ratings_meta.json").read_text())
assert meta["status"] == "ok", f"статус L6: {meta['status']}"
assert meta.get("display_rule"), "правило отображения меток не применено (scripts/apply_display_rules.py)"
low_minutes = ratings[ratings.toi_min < 60]
assert (low_minutes.reliability == "LOW").all(), "все игроки с TOI < 60 мин должны быть LOW"
rows, boot = meta["rows"], meta["model_bootstrap"]
print("определение:", meta.get("rating_definition"), "| бутстрэп:", boot.get("status"), f'B = {meta.get("B_done")}/{meta.get("B")}')
print("метки в файле:", meta["display_rule"]["counts_display"], "| метки модели:", meta["display_rule"]["counts_model"],
      "| понижено «мало минут»:", meta["display_rule"]["n_demoted"])
print("вырожденных интервалов:", rows["n_degenerate_intervals"], "| средняя полуширина:", round(rows["mean_halfwidth"], 4))

## Проверка: устойчивость, предсказание будущей игры, сравнение с базовыми показателями

In [ ]:
for name in ("validation_report.md", "baseline_report.md"):
    path = OUT / name
    print(path.read_text() if path.exists() else f"{name}: ещё не построен (scripts/{name.split('.')[0]}.py)")

## Границы применимости

Кратко (подробно — `docs/METHODOLOGY.md`, раздел 8): оценка сравнивает игроков внутри амплуа; у игроков
соперников мало матчей, поэтому их оценки сильно стянуты к среднему амплуа (смотрите `reliability` и ширину
интервала); партнёры по звену в уровне A не разделяются; данные — только моменты событий, без трекинга.

## Карточка игрока: WAR, роли, ситуации, удаления, физический профиль

Отдельные поля вне рейтинга (METHODOLOGY 11): всё выбирается по матчам разработки train, test только применяется; `test_ratings.csv` не меняется. Итог — `outputs/card/card_report.md` и сводная таблица «Авангарда».

In [ ]:
env = dict(os.environ, PYTHONPATH=str(ROOT / "src"))
for script in ("card_war", "card_roles", "card_situational", "card_penalties", "card_phys", "card_faceoffs", "card_club", "card_report"):
    print("$", script, flush=True)
    subprocess.run([str(PY), f"scripts/{script}.py"], cwd=ROOT, env=env, check=True)
print((OUT / "card" / "card_report.md").read_text()[:4000])

## Проверка воспроизводимости

sha256 итогового `test_ratings.csv` должен совпасть с записью в метаданных прогона; рядом — эталон сданной версии.

In [ ]:
import hashlib
REFERENCE_SHA = "eec6df1d194787ac3ac18dc6022f696a72e2a5e5ca98fa3884a0287157a8efaa"  # сданная версия 07–08.10
sha = hashlib.sha256((OUT / "test_ratings.csv").read_bytes()).hexdigest()
meta_sha = json.loads((OUT / "test_ratings_meta.json").read_text())["csv_sha256"]
assert sha == meta_sha, f"test_ratings.csv не совпадает с метаданными прогона: {sha} != {meta_sha}"
print("sha256:", sha)
print("совпадает с эталоном сданной версии:", sha == REFERENCE_SHA)